# Prompting Fundamentals- Directing a Brilliant, Literal-Minded Performer

**Module 5 · Lesson 5.1**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Prompt anatomy - the four parts of every prompt
- Instruction clarity - be specific, or get the average
- Roles and system prompts - tell the model who it is
- Delimiters and structured output - separate data from instructions
- Reasoning and parameters - classic models vs reasoning models
- Failure modes and reliability - it will lie; measure before you ship

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## The whole game: same model, two prompts, two different worlds

> 🎤 **Analogy**
>
> **You are directing a brilliant actor.** Tell them *"just do something emotional"* and you get a mediocre take. Tell them *"you have just learned your childhood friend is alive after ten years; you are in a crowded train station; start with disbelief, then let joy overwhelm you"* - and the scene is unforgettable. Same actor. Different direction. Completely different result.
> A frontier model is exactly that performer: enormously capable, perfectly literal, and only as good as the direction you give it. **Prompting is direction.** By the end of this lesson you will take a free-text product review and turn it into clean, validated, structured data with a prompt you have *measured*.
> **Where the analogy breaks down:** unlike an actor, the model has no memory of the last scene unless you put it in the context, no shared backstory, and it will confidently improvise false details rather than ask. That gap is exactly why grounding and reliability testing exist (Step 6).

Everything here is tested with **real API calls** - no technique without proof. You will see the weak prompt, the strong prompt, and the difference side by side. We use Google's Gemini through the current unified SDK, but every idea transfers to Claude (Anthropic) and GPT (OpenAI); where providers differ, we say so.

- **Analyze** any prompt into its four parts - instruction, context, input, output format - and diagnose which part a weak prompt is missing.

- **Apply** the core techniques - specificity, role and system prompts, delimiters, output formatting, zero-shot reasoning - against the live API.

- **Contrast** classic-model prompting with reasoning-model prompting, and choose the right style for the model in front of you.

- **Evaluate** prompt reliability empirically (run it N times, measure the pass rate) and defend against hallucination and sycophancy.

## Warm-up: 60 seconds of recall

Tap each card to check yourself against earlier lessons. Each one is load-bearing for today.

## The setup: one helper we will reuse all lesson

Every example below calls this `ask()` helper. It uses the current, unified **google-genai** SDK. Google deprecated the older `google.generativeai` package on 2025-08-31, so any tutorial still importing it is out of date ([migration guide](https://ai.google.dev/gemini-api/docs/migrate)).

**📝 `setup.py Gemini`** - *API*

In [ ]:
# pip install "google-genai>=1.0.0"   (the unified SDK; replaces google.generativeai)
from google import genai
from google.genai import types
import os, time

# One client for the whole program. Reads GOOGLE_API_KEY from the environment.
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

def ask(prompt: str, temperature: float = 0.3) -> str:
    """Send one prompt to Gemini and return the text. Retries once on a transient error."""
    for attempt in range(2):
        try:
            resp = client.models.generate_content(
                model="gemini-3.5-flash",
                contents=prompt,
                config=types.GenerateContentConfig(temperature=temperature),
            )
            return resp.text
        except Exception as e:
            if attempt == 1:
                return f"[API error after retry: {e}]"
            time.sleep(1.0)   # back off before retrying; real code retries ONLY transient 429/503/timeout, with exponential backoff

print(ask("Say hello in one word."))
# Output: Hello

- `from google import genai` - the unified package. The old `import google.generativeai as genai` still appears in 2024 tutorials; it is deprecated and will eventually stop receiving updates.

- `genai.Client(...)` - you build one client object and reuse it, instead of the old `genai.configure()` global plus a per-call `GenerativeModel`.

- `client.models.generate_content(model=..., contents=..., config=...)` - the single call. The model id is passed per call, so switching models is a one-word change.

- `types.GenerateContentConfig(...)` - where sampling settings (and later the system instruction) live. We will pass `system_instruction` here in Step 3.

- The `try/except` with one retry is the smallest honest production pattern - network calls fail, and a helper that crashes the whole script on a blip is not shippable.

Default model: `gemini-3.5-flash` (Google's current fast tier, released Dec 2025). The exact id and price move; verify against [ai.google.dev](https://ai.google.dev/gemini-api/docs/migrate) before you ship.

---

## 🎯 Concept 1: Prompt anatomy - the four parts of every prompt

### Prompt anatomy - the four parts of every prompt

The same skeleton underneath Gemini, Claude, and GPT prompts alike.

**A prompt is a text message to a new intern who cannot ask follow-up questions.** "Do the thing" gets you a shrug. "Email the 3 Mumbai clients a one-line reminder that invoices are due Friday" gets you the work done.

The gain: four pieces of information turned a shrug into a finished task - who/why (context), what (instruction), on what (input), in what shape (format). Rough, but it is the whole idea.

> 📨 **Analogy**
>
> **A prompt is like a job posting.** A good posting states what you need done (instruction), the background (context), the specific requirement (input), and how to respond (output format). Drop any part and you get irrelevant applicants - or, for an LLM, irrelevant text.
> **Where it breaks down:** a human applicant fills gaps by asking or using common sense. The model does neither - it fills gaps by guessing the statistically average answer, which is why a vague prompt feels so generic.

**📝 `anatomy.py`** - *Conceptual*

In [ ]:
# THE 4 PARTS OF EVERY PROMPT
#   1 INSTRUCTION   - what to do
#   2 CONTEXT       - who it is for, why, any background
#   3 INPUT         - the specific thing to act on
#   4 OUTPUT FORMAT - the exact shape of the answer

bad = "Tell me about Hyderabad"          # instruction only

good = """[INSTRUCTION] Write a travel-guide paragraph.
[CONTEXT]     For a first-time US visitor on a 3-day trip.
[INPUT]       City: Hyderabad, India.
[OUTPUT]      Top 3 attractions, best food, one insider tip. Under 150 words."""

print(ask(good))
# Output: Hyderabad packs history and flavor into three days. Start at the
# Output: Charminar and Golconda Fort... (a tight, on-brief 140-word guide)

#### 💡 What just happened

⚡ What just happened? The four labels are not magic syntax - they are a checklist. The bad prompt had only an instruction, so the model guessed at the other three. The good prompt pinned all four, so there was nothing left to guess. **Anthropic's colleague test:** if a new teammate could not act on your prompt with zero extra context, the model cannot either.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

That was a teaching prompt. Production prompts wear the same four parts but look less tidy - the instruction up top, the user's data fenced off, the format pinned at the end. Here is the same anatomy inside a real support-bot prompt:

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Instruction clarity - be specific, or get the average

### Instruction clarity - be specific, or get the average

The single most common prompting mistake, and the cheapest one to fix.

**Ordering food.** "Give me something good" might get you biryani when you wanted pizza. "One medium Margherita, thin crust, no olives, cut into 8" gets you exactly that.

The gain: every extra constraint is one fewer thing the kitchen has to guess. A prompt is an order; vagueness is just unspecified choices the model makes for you.

Let us prove it. Same task, vague versus specific, tested through the API:

**📝 `clarity.py Gemini`** - *API*

In [ ]:
vague = "Tell me about machine learning."

specific = """Explain machine learning to a 15-year-old who has never heard of it.
Use exactly 3 sentences. Include one example they would relate to (Instagram
or YouTube). Do not use any technical jargon."""

print("=== VAGUE ==="); print(ask(vague))
print("=== SPECIFIC ==="); print(ask(specific))
# Output: === VAGUE ===   a 250-word textbook block, generic, no audience
# Output: === SPECIFIC === Machine learning is how apps learn what you like.
# Output: When Instagram shows you reels you enjoy, it learned from your taps...
# Output: (exactly 3 sentences, an Instagram example, zero jargon)

#### 💡 What just happened

⚡ What just happened? We added four constraints - audience, length, a concrete example, and a register rule - and every one showed up. **Specificity is not politeness; it is control.** Each constraint narrows the distribution of plausible answers toward the one you actually want.

> 📦 **Info**
>
> The 5 levers of a clear instruction
> - **Audience** - "explain to a 5-year-old" vs "to a PhD reviewer" rewrites the whole answer.
> - **Length** - "in 2 sentences", "in exactly 100 words", "as 5 bullets". Without it, the model guesses.
> - **Say what TO do and what NOT to do** - "explain the idea; do not include code". Negative constraints are surprisingly effective.
> - **Show the shape** - "format each line as Term: definition (example)". A tiny example of the format beats a paragraph describing it.
> - **Decompose** - "first identify the topic, then list 3 points, then summarize". Numbered structure raises quality on multi-part tasks.

> 💡 **Info**
>
> ⚠️ Common mistake Beginners think "be more detailed" or "be accurate" are constraints. They are not - they are wishes. The model cannot act on "be accurate"; it can act on "answer only from the text below, and quote the sentence you used." Replace adjectives with rules.

---

## 🎯 Concept 3: Roles and system prompts - tell the model who it is

### Roles and system prompts - tell the model who it is

The same question to "a junior intern" and "a senior engineer" yields very different answers.

**A great actor plays many characters.** "Play a stern professor" and "play a goofy comedian" change the same actor's tone, vocabulary, and pacing.

The gain: a role is one sentence that shifts everything downstream. It does not give the model new knowledge - it picks which of its existing knowledge to surface.

> 🎭 **Analogy**
>
> **Setting a role is casting.** "You are a senior data scientist" shifts the model's vocabulary, depth, and even its default assumptions toward how such a person writes. The behaviour change is real and large.
> **Where it breaks down:** casting does not grant competence. A role makes the model *sound* like a cardiologist; it does not make medical claims correct. Roles steer style and emphasis - they are not a substitute for grounding (Step 6).

In production you do not paste the role into every message - you set it once in the **system prompt**, where it conditions every following turn (this is the same "persona lives at the front of the context" idea you met in Module 4):

**📝 `system_prompt.py Gemini`** - *API*

In [ ]:
from google import genai
from google.genai import types
import os

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# The role is set ONCE in the config and applies to every call that uses it.
config = types.GenerateContentConfig(
    system_instruction=(
        "You are a senior backend engineer with 12 years building production "
        "Python APIs that serve millions of requests a day. Answer with the "
        "patterns you would actually ship: be concrete, name the failure modes, "
        "and keep it under 150 words."
    ),
    temperature=0.3,
)

resp = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="How should I handle errors in my Python API?",
    config=config,
)
print(resp.text)
# Output: Wrap handlers in typed exceptions, not bare except. Return structured
# Output: error envelopes with a code + request_id. Add timeouts and bounded
# Output: retries with backoff. Log with context; emit metrics; alert on 5xx rate...

- The system prompt lives in `types.GenerateContentConfig(system_instruction=...)`, not in a `GenerativeModel(system_instruction=...)` constructor as the deprecated package did.

- You build the `config` once and pass it to many `generate_content` calls - the role is reused without repeating it in every message.

- System instruction versus user content: the role and rules go in `system_instruction`; the actual task and data go in `contents`. Keeping them separate is also your first injection defense (Step 4).

#### 💡 What just happened

⚡ What just happened? One sentence of casting turned a generic try/except tutorial into shippable advice: typed exceptions, structured error envelopes, bounded retries, metrics. The role did not teach the model anything new - it selected which slice of what it already knows to write from. **This is how every customer-support bot, tutor, and coding assistant gets a consistent voice.**

---

## 🎯 Concept 4: Delimiters and structured output - separate data from instructions

### Delimiters and structured output - separate data from instructions

The first defense against injection, and the way to get JSON your code can actually use.

**Quotation marks.** "Translate the phrase good morning" is ambiguous; *Translate the phrase "good morning"* is not. The quotes mark where the data starts and ends.

The gain: delimiters are industrial-strength quotation marks. They draw a hard border between your instructions and the user's content.

Two jobs, one technique. First, delimiters fence off untrusted input. Providers differ in what they are tuned for:

**📝 `delimiters.py`** - *Multi-provider*

In [ ]:
# Claude is tuned toward XML tags; GPT toward markdown headers/fences. Gemini takes either.
claude_style = """Summarize the document in 3 sentences.
<document>
{user_text}
</document>"""

gpt_style = """### Instructions
Summarize the document in 3 sentences.
### Document
```
{user_text}
```"""

# Either way, the model now treats the fenced region as DATA, not commands.
print("Delimiters drawn: instructions and user data are now separated.")
# Output: Delimiters drawn: instructions and user data are now separated.

Second, the same fencing instinct gives you **structured output** - show the exact shape and the model fills it in. This is the prompt-level version of structured output; native JSON mode and tool schemas are Lesson 5.5's job.

**📝 `json_output.py Gemini`** - *API*

In [ ]:
import json

review = """Bought the Sony WH-1000XM5 for 29,999 rupees on Amazon. The noise
cancelling is incredible. Battery lasts forever. The touch controls are fiddly. 4/5."""

prompt = f"""Extract structured data from the review below.
Return ONLY valid JSON (no markdown, no commentary) with this exact shape:
{{"product": str, "price": number|null, "rating": number, "sentiment": "positive"|"negative"|"mixed", "cons": [str]}}

<review>
{review}
</review>"""

raw = ask(prompt, temperature=0.0).strip()   # temp 0 for stable, parseable output
if raw.startswith("```"):                        # defensive: strip a stray ```json fence before parsing
    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
data = json.loads(raw)
print(data["product"], "|", data["rating"], "|", data["sentiment"])
# Output: Sony WH-1000XM5 | 4 | mixed

- **Shape shown, not described** - giving the literal JSON skeleton (keys, types, enum values) teaches the format far better than a sentence describing it.

- **"ONLY valid JSON, no markdown"** - without it, models love to wrap output in ````json ... ```` fences that break `json.loads`.

- **temperature 0.0** - structured extraction wants determinism, not creativity; randomness here only invents new key names.

- **Fenced input** - the `<review>` tags keep any instruction-like text inside the review from hijacking the extraction.

#### 💡 What just happened

⚡ What just happened? One technique, two payoffs. Delimiters separate *your* instructions from *their* data (the foundation of injection defense, deepened in Lesson 5.7), and showing the exact output shape turns free text into JSON your code can consume. **Free text in, structured data out** is the backbone of nearly every production LLM feature.

---

## 🎯 Concept 5: Reasoning and parameters - classic models vs reasoning models

### Reasoning and parameters - classic models vs reasoning models

The 2026 shift: the old "think step by step" advice can now backfire.

**Two kinds of helper.** A junior who blurts the first thing benefits from "slow down, show your working." A senior who already reasons carefully does not need you narrating their method - just tell them the goal.

The gain: match the instruction to the worker. Classic models are the junior; reasoning models are the senior.

On a **classic** instruction-following model, *zero-shot* chain-of-thought (zero-shot = you give the model no worked examples in the prompt, just the task; adding examples is *few-shot*, the subject of Lesson 5.2) is a real, measured win. Kojima et al. (2022) found that simply adding "Let's think step by step" lifted GSM8K math accuracy from roughly 10.4% to 40.7% (about a fourfold jump) - but that was measured on *non-reasoning* models, and that caveat is the whole point in 2026.

**📝 `classic_cot.py Gemini`** - *API*

In [ ]:
# CLASSIC MODEL: zero-shot chain-of-thought helps on multi-step math.
direct = "Shirt is 800 rupees, 25% off, then a 100-rupee coupon. Final price?"
cot    = direct + " Let's think step by step."

print(ask(cot, temperature=0.0))
# Output: 800 - 25% = 600. 600 - 100 coupon = 500. Final price: 500 rupees.

On a **reasoning** model, the move is the opposite: state the goal and the constraints, and get out of the way. Process narration is the noise, not the signal.

The canonical 2026 wrong move - everything the 2023 playbook taught, piled onto a model that already reasons:

On a reasoning model this preamble adds tokens, narrows the search space, and can make the answer *more* mechanical and occasionally worse. The fix is the outcome-first version: *"Final price after the discount and coupon on the 800-rupee shirt? Answer with the number only."*

The second knob in this step is **sampling temperature** - how sharp or flat the next-token distribution is before a token is drawn.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

**📝 `temperature.py Gemini`** - *API*

In [ ]:
prompt = "Write a tagline for a Hyderabad biryani restaurant."

for t in (0.0, 0.7, 1.4):
    print(t, "->", ask(prompt, temperature=t))
# Output: 0.0 -> Authentic Hyderabadi biryani, every single day.   (safe, repeatable)
# Output: 0.7 -> Dum-cooked tradition, served by the kilo.        (balanced, default)
# Output: 1.4 -> Saffron rebellion in a copper handi!            (creative, riskier)

> 📦 **Info**
>
> Temperature as a heuristic, not a lawRough vendor-guidance starting points: **0.0** for classification, extraction, and code; **0.2-0.5** for summarization; **0.7-1.0** for creative writing. These are dials to tune per task, not fixed truths. And note: temperature controls *randomness*, not correctness - it cannot make a wrong answer right or stop a hallucination.

Now contrast the two prompting styles directly:

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?"Think step by step" is foundational vocabulary, but in 2026 it is no longer universal advice. On a classic model it adds real accuracy; on a reasoning model it can subtract it. The skill is reading which model is in front of you and matching the style - process-heavy for classic, outcome-first for reasoning. Advanced reasoning, self-consistency, and extended thinking are Lesson 5.3.

---

## 🎯 Concept 6: Failure modes and reliability - it will lie; measure before you ship

### Failure modes and reliability - it will lie; measure before you ship

Hallucination, sycophancy, and the run-it-N-times test that turns "it worked once" into evidence.

**A confident friend who never says "I don't know."** Ask for a restaurant's exact closing time and they will give you one - right or not - because guessing feels more helpful than admitting ignorance.

The gain: the cure is not "try harder", it is "here is the menu - answer only from it, and say so if it is not there." That is grounding.

Two failure modes bite production hardest:

> 📦 **Info**
>
> 1. Hallucination - confident, plausible, false
> The model invents facts that sound right. It is not a crash and temperature 0 does not stop it ([OpenAI, 2025](https://openai.com/index/why-language-models-hallucinate/)). **Defense: grounding.** Provide reference text, instruct "answer only from the text below; if it is not there, say 'not found'", and ask it to quote the supporting line before answering.

> 💡 **Info**
>
> 2. Sycophancy - agreeing with a confidently wrong user
> Models tend to side with the user even when the user is wrong (Sharma et al., 2023). This is not hypothetical: in April 2025 OpenAI shipped an over-agreeable GPT-4o update and **rolled it back within days**. **Defense:** ask the model to reason independently first, then compare to the user's claim - and never evaluate a model by "did it agree with me?"

> 📦 **Info**
>
> A global reminder: domain and language gapsEven frontier models are uneven across the world's languages. On the IndicParam benchmark (Dec 2025) the best model averaged only about 58%, and none exceeded ~41.5% on low-resource languages like Bodo and Santali. The lesson for engineers building worldwide: **always provide reference text and test on your real users' languages** - do not assume English-level quality everywhere.

The discipline that ties it together: a prompt is code, so **test it like code**. Run it N times and measure the pass rate instead of eyeballing one lucky run.

**📝 `reliability_harness.py Gemini`** - *API*

In [ ]:
import json

def measure(prompt, checks, runs=10):
    """Run a prompt many times; report how often each check passes."""
    passes = {name: 0 for name in checks}
    for _ in range(runs):
        out = ask(prompt, temperature=0.7)   # >0 so runs actually vary - that variance is what we measure
        for name, check in checks.items():
            try:
                if check(out): passes[name] += 1
            except Exception:
                pass   # a thrown check counts as a fail
    return {name: f"{c}/{runs}" for name, c in passes.items()}

prompt = 'Return ONLY JSON: {"sentiment":"positive"|"negative","rating":1-5}. ' \
         'Review: "Great battery, dim screen. 3 stars."'

checks = {
    "valid_json": lambda r: json.loads(r) is not None,
    "has_rating": lambda r: "rating" in json.loads(r),
    "rating_is_3": lambda r: json.loads(r).get("rating") == 3,
}
print(measure(prompt, checks))
# Output: {'valid_json': '10/10', 'has_rating': '10/10', 'rating_is_3': '8/10'}
# The 8/10 is the real signal: sometimes the model reads "3 stars" as mixed=2.5.

#### 💡 What just happened

⚡ What just happened? One run tells you nothing; ten runs tell you the truth. The harness turns "it worked when I tried it" into a pass rate you can gate a deploy on. **The default engineering assumption: the model will hallucinate and occasionally drift unless you ground it and measure it.** Systematic eval-driven development scales this up in Module 14.

## Putting it together: the Product Review Analyzer

Now combine the lesson into one production-shaped tool: a **role** (system prompt) plus **clear instructions** plus **structured output** validated by Pydantic. Free-text reviews go in; clean, typed, validated data comes out.

**📝 `review_analyzer.py Complete`** - *project*

In [ ]:
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
import os, json

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# 1. The exact output contract, as a Pydantic model.
class ReviewAnalysis(BaseModel):
    product: str
    rating: int = Field(ge=1, le=5)
    sentiment: str                # positive | negative | mixed
    pros: list[str]
    cons: list[str]
    recommend: bool

# 2. The role lives in the system instruction; set once.
config = types.GenerateContentConfig(
    system_instruction=(
        "You are a precise e-commerce review analyst. You output ONLY valid JSON "
        "matching the requested schema - no markdown fences, no commentary."
    ),
    temperature=0.0,
)

def analyze(review: str) -> ReviewAnalysis:
    prompt = f"""Extract data from the review. Return ONLY JSON matching:
{json.dumps(ReviewAnalysis.model_json_schema())}
Rules: rating is 1-5; sentiment is positive|negative|mixed; recommend is true if rating >= 3.
<review>{review}</review>"""
    for attempt in range(2):                       # 3. validate, and retry once on bad JSON
        try:
            raw = client.models.generate_content(
                model="gemini-3.5-flash", contents=prompt, config=config).text.strip()
            if raw.startswith("```"):
                raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
            return ReviewAnalysis.model_validate_json(raw)
        except Exception:
            if attempt == 1: raise

r = analyze("OnePlus Buds 3, 3299 rupees. Great bass and ANC, but they fall out when I jog. 4 stars.")
print(r.product, "|", r.rating, "|", r.sentiment, "|", r.recommend)
# Output: OnePlus Buds 3 | 4 | mixed | True

#### 💡 What just happened

⚡ What just happened? Every technique from the lesson, in one function: a **role** (system prompt), **clear rules**, **delimited input**, **structured output**, and a **validation-plus-retry** loop so malformed JSON never reaches your database. This free-text-in, typed-data-out pattern is exactly how e-commerce platforms process reviews at scale - and it is the payoff promised in the cold open.

### 🧮 The whole lesson on one screen

**Anatomy** (4 parts) gives you the skeleton; **clarity** removes the model's room to wander; a **role** selects which knowledge it writes from; **delimiters and structure** separate data from instructions and make output machine-readable; the **classic-vs-reasoning** split tells you whether to narrate the process or just name the goal; and **reliability** reminds you the model will hallucinate unless you ground it and measure it.

Forward hooks you just planted: **5.2** few-shot and in-context learning, **5.3** advanced reasoning and self-consistency, **5.4** context engineering, **5.5** native structured output and tool use, **5.6** prompt optimization and DSPy, **5.7** adversarial prompting and injection defenses. Systematic evaluation scales up in **Module 14**.

- Schulhoff et al., *The Prompt Report: A Systematic Survey of Prompting Techniques* (2024) - [arxiv.org/abs/2406.06608](https://arxiv.org/abs/2406.06608)

- Kojima et al., *Large Language Models are Zero-Shot Reasoners* (2022; the "let's think step by step" result) - [arxiv.org/abs/2205.11916](https://arxiv.org/abs/2205.11916)

- Kalai et al. (OpenAI), *Why Language Models Hallucinate* (2025) - [arxiv.org/abs/2509.04664](https://arxiv.org/abs/2509.04664) and the [plain-language explainer](https://openai.com/index/why-language-models-hallucinate/)

- Sharma et al. (Anthropic), *Towards Understanding Sycophancy in Language Models* (2023) - [arxiv.org/abs/2310.13548](https://arxiv.org/abs/2310.13548); OpenAI [GPT-4o sycophancy rollback](https://openai.com/index/sycophancy-in-gpt-4o/) (Apr 2025)

- OpenAI GPT-5 / GPT-5.5 prompting guides ("write for outcomes, not ritual") - [developers.openai.com](https://developers.openai.com/cookbook/examples/gpt-5/gpt-5_prompting_guide)

- Anthropic prompt-engineering and XML-tag docs - [platform.claude.com](https://platform.claude.com/docs/en/build-with-claude/prompt-engineering/use-xml-tags)

- Google unified `google-genai` SDK migration guide - [ai.google.dev/gemini-api/docs/migrate](https://ai.google.dev/gemini-api/docs/migrate)

- IndicParam benchmark (low-resource Indic gap, Dec 2025) - [arxiv.org/abs/2512.00333](https://arxiv.org/abs/2512.00333)

## Key takeaways

> ℹ️ **Info**
>
> What you learned - 6 concepts
> - **Prompt anatomy** - instruction, context, input, output format. Diagnose a weak prompt by the part it is missing.
> - **Instruction clarity** - replace adjectives ("be accurate") with rules ("answer only from the text; quote the line").
> - **Roles and system prompts** - cast the model once in the system instruction; it selects which knowledge to surface.
> - **Delimiters and structured output** - fence data from instructions (first injection defense) and show the exact output shape.
> - **Classic vs reasoning** - "think step by step" helps classic models and can hurt reasoning models; write for outcomes.
> - **Reliability** - the model will hallucinate; ground it with reference text and measure the pass rate over N runs before shipping.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Prompting Fundamentals- Directing a Brilliant, Literal-Minded Performer**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-5_1.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-5.1.html` - regenerate this notebook whenever the source HTML is updated.*
